# 21 · Hybrid CTC/AED → Qwen3 LoRA translation evaluation

20번의 저장된 `hybrid_best.pt`와 18번의 영어·한국어 robustness LoRA를 연결합니다. **학습 셀 없이 새 Colab GPU 런타임에서 위에서부터 실행**하세요.

- Drive의 기존 `S2ST` 폴더, Kimi feature shard 5개, Hybrid checkpoint, LoRA adapter 2개가 필요합니다. 원본 ipynb는 런타임에 업로드할 필요가 없습니다.
- 18번의 `character_ctc_transcriptions.csv`에 있는 **동일 300개·동일 순서·동일 번역 reference**를 사용합니다.
- Oracle / Character CTC / Hybrid greedy / Hybrid beam1 / Hybrid rescored를 같은 LoRA로 비교합니다. 프롬프트, generation, chrF++와 BLEU는 18번과 동일합니다.
- 번역 전에 Myanmar 전사 하나를 확정합니다. Reference는 후보 선택에 사용하지 않습니다. N-best oracle은 진단 지표입니다.
- 이 노트북은 음성 feature → Myanmar 전사 → 영어/한국어 **텍스트 번역**까지입니다. TTS와 음성 출력 평가는 포함하지 않습니다.
- 20번의 dev에서 선택한 모델을 18번의 held-out subset에서 비교하므로 독립 test 성능이라고 해석하지 않습니다. 이 300개에서 가중치를 조정하지 마세요.

기본 경로가 다르면 다음 설정 셀만 수정하세요. 4B LLM을 비양자화 상태로 로딩하므로 GPU VRAM 16GB 이상을 권장하며, ASR 모델은 LLM 로딩 전에 해제합니다.


In [1]:
# 새 런타임에서 가장 먼저 실행. 이미 transformers를 import한 런타임이면 재시작 후 실행.
%pip -q uninstall -y torchao
%pip -q install "transformers==4.57.3" "peft==0.17.1" accelerate safetensors pandas numpy sacrebleu jiwer tqdm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, re, math, random, gc, json, hashlib, unicodedata, importlib.metadata
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig
from sacrebleu.metrics import CHRF, BLEU
from IPython.display import display

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Colab: 런타임 → 런타임 유형 변경 → GPU를 선택하세요.'
torch.cuda.manual_seed_all(SEED)
DEVICE='cuda'; DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
AMP_DTYPE=DTYPE
ROOT=Path('/content/drive/MyDrive/S2ST')
OLD_DIR=ROOT/'baselines/bigru_prefix_semantic_alignment'
FEATURE_DIR=OLD_DIR/'kimi_features'
MANIFEST_PATH=OLD_DIR/'manifest.csv'
SPLIT_PATH=OLD_DIR/'train_dev_split.csv'
HYBRID_CKPT=ROOT/'baselines/hybrid_conformer_ctc_aed/hybrid_best.pt'
LORA_ROOT=ROOT/'baselines/qwen3_translation_lora'
EVAL_PATH=LORA_ROOT/'character_ctc_eval/character_ctc_transcriptions.csv'
ADAPTERS={'en':LORA_ROOT/'english_character_ctc_robustness_lora',
          'ko':LORA_ROOT/'korean_character_ctc_robustness_lora'}
LLM_ID='Qwen/Qwen3-4B-Instruct-2507'
OUT_DIR=ROOT/'baselines/hybrid_ctc_aed_lora_eval_21'
OUT_DIR.mkdir(parents=True,exist_ok=True)
# 20번의 고정 decoding 값. 평가 300개를 보고 튜닝하지 마세요.
BEAM_SIZE=10; N_BEST=10; AED_RESCORE_WEIGHT=.35; LENGTH_BONUS=.05
# ASR은 길이/padding 영향 없이 샘플별 추론. LLM은 OOM 시 배치를 자동으로 줄입니다.
LLM_BATCH_SIZE=8
RESUME=True
print(torch.cuda.get_device_name(), DTYPE)
for p in [HYBRID_CKPT, MANIFEST_PATH, EVAL_PATH, *ADAPTERS.values()]:
    assert p.exists(), f'필요한 파일/폴더가 없습니다: {p}'
for lang,p in ADAPTERS.items():
    assert (p/'adapter_config.json').exists(), p
    assert any((p/n).exists() for n in ['adapter_model.safetensors','adapter_model.bin']), p
    cfg=PeftConfig.from_pretrained(str(p))
    assert cfg.base_model_name_or_path==LLM_ID, (lang,cfg.base_model_name_or_path,LLM_ID)

Mounted at /content/drive
NVIDIA RTX PRO 6000 Blackwell Server Edition torch.bfloat16


In [3]:
def clean_text(x):
    if x is None or (isinstance(x,float) and np.isnan(x)): return ''
    return ' '.join(unicodedata.normalize('NFC',str(x)).strip().split())
def norm_text(x): return unicodedata.normalize('NFC',str(x)).strip()
def atomic_csv(df,path):
    tmp=Path(str(path)+'.tmp'); df.to_csv(tmp,index=False); os.replace(tmp,path)
def file_sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()

eval_ctc=pd.read_csv(EVAL_PATH,dtype={'sample_id':str},keep_default_na=False)
required=['sample_id','source','transcription_my','character_ctc_text','translation_en','translation_ko']
assert set(required)<=set(eval_ctc), set(required)-set(eval_ctc)
assert len(eval_ctc)==300 and eval_ctc.sample_id.is_unique
for c in required:
    assert eval_ctc[c].map(lambda x:isinstance(x,str)).all(), c
for c in ['sample_id','transcription_my','translation_en','translation_ko']:
    assert eval_ctc[c].str.strip().ne('').all(), c
manifest=pd.read_csv(MANIFEST_PATH,dtype={'sample_id':str},keep_default_na=False)
assert manifest.sample_id.is_unique
if 'split' not in manifest:
    splits=pd.read_csv(SPLIT_PATH,dtype={'sample_id':str})
    manifest=manifest.merge(splits[['sample_id','split']],on='sample_id',validate='one_to_one')
assert set(eval_ctc.sample_id)<=set(manifest.sample_id)
matched=manifest.set_index('sample_id').loc[eval_ctc.sample_id]
assert matched.split.eq('dev').all(), '평가 ID가 train split에 있습니다.'
for c in ['source','transcription_my','translation_en','translation_ko']:
    assert c in matched
    assert [clean_text(x) for x in matched[c]]==[clean_text(x) for x in eval_ctc[c]], f'reference/source 불일치: {c}'
# 체크포인트의 실제 vocab을 사용합니다. manifest에서 vocabulary를 재생성하지 않습니다.
ckpt=torch.load(HYBRID_CKPT,map_location='cpu',weights_only=False)
assert {'model','config'}<=set(ckpt)
id_to_char=ckpt['config']['vocab']; INPUT_DIM=int(ckpt['config']['input_dim'])
assert isinstance(id_to_char,list) and id_to_char[:4]==['<blank>','<unk>','<bos>','<eos>']
assert len(set(id_to_char))==len(id_to_char)
special=id_to_char[:4]; char_to_id={c:i for i,c in enumerate(id_to_char)}
BLANK,UNK,BOS,EOS=range(4)
def decode_text(ids): return ''.join(id_to_char[i] for i in ids if i>=len(special))

# 캐시가 다른 모델/설정의 결과와 섞이지 않도록 내용 hash를 기록합니다.
provenance={'hybrid_sha256':file_sha256(HYBRID_CKPT),'eval_sha256':file_sha256(EVAL_PATH),
 'llm_id':LLM_ID,'beam_size':BEAM_SIZE,'n_best':N_BEST,'aed_weight':AED_RESCORE_WEIGHT,
 'length_bonus':LENGTH_BONUS,'asr_batch_size':1,'dtype':str(DTYPE),'seed':SEED,
 'prompt_generation':'18_exact_v1','pipeline_version':'21_v1',
 'packages':{n:importlib.metadata.version(n) for n in ['torch','transformers','peft','sacrebleu']},
 'adapters':{lang:{p.name:file_sha256(p) for p in sorted(folder.iterdir()) if p.is_file()}
             for lang,folder in ADAPTERS.items()}}
RUN_HASH=hashlib.sha256(json.dumps(provenance,sort_keys=True).encode()).hexdigest()[:16]
RUN=OUT_DIR/RUN_HASH; RUN.mkdir(exist_ok=True)
(RUN/'provenance.json').write_text(json.dumps(provenance,ensure_ascii=False,indent=2))
atomic_csv(eval_ctc[required],RUN/'evaluation_manifest.csv')
print('epoch:',ckpt.get('epoch'),'vocab:',len(id_to_char),'run:',RUN)
display(eval_ctc.groupby('source').size().rename('samples'))

epoch: 9 vocab: 168 run: /content/drive/MyDrive/S2ST/baselines/hybrid_ctc_aed_lora_eval_21/f9f894e6596c1503


,samples
source,
fleurs,100
mig-burmese-audio-transcription,100
openslr,100


In [4]:
class ConvModule(nn.Module):
    def __init__(self,d,k=15):
        super().__init__(); self.n=nn.LayerNorm(d)
        self.pw1=nn.Conv1d(d,2*d,1); self.dw=nn.Conv1d(d,d,k,padding=k//2,groups=d)
        self.bn=nn.BatchNorm1d(d); self.pw2=nn.Conv1d(d,d,1)
    def forward(self,x,mask=None):
        z=self.n(x).transpose(1,2); a,b=self.pw1(z).chunk(2,1)
        z=a*torch.sigmoid(b); z=self.pw2(F.silu(self.bn(self.dw(z)))).transpose(1,2)
        if mask is not None: z=z.masked_fill(mask.unsqueeze(-1),0)
        return x+z

class ContextBlock(nn.Module):
    def __init__(self,d=384,h=6,ff=1536,drop=.1):
        super().__init__(); self.ff1=nn.Sequential(nn.LayerNorm(d),nn.Linear(d,ff),nn.SiLU(),nn.Dropout(drop),nn.Linear(ff,d))
        self.n=nn.LayerNorm(d); self.attn=nn.MultiheadAttention(d,h,dropout=drop,batch_first=True)
        self.conv=ConvModule(d); self.ff2=nn.Sequential(nn.LayerNorm(d),nn.Linear(d,ff),nn.SiLU(),nn.Dropout(drop),nn.Linear(ff,d)); self.out=nn.LayerNorm(d)
    def forward(self,x,mask):
        x=x+.5*self.ff1(x); z=self.n(x); x=x+self.attn(z,z,z,key_padding_mask=mask,need_weights=False)[0]
        x=self.conv(x,mask); return self.out(x+.5*self.ff2(x))

class AttentionDecoder(nn.Module):
    def __init__(self,vocab,d=384,emb=256):
        super().__init__(); self.emb=nn.Embedding(vocab,emb); self.rnn=nn.GRU(emb+d,d,batch_first=True)
        self.q=nn.Linear(d,d,bias=False); self.out=nn.Linear(2*d,vocab)
    def attend(self,h,memory,mem_mask):
        e=torch.bmm(self.q(h),memory.transpose(1,2))/math.sqrt(memory.size(-1))
        e=e.masked_fill(mem_mask.unsqueeze(1),-1e4); a=e.softmax(-1); return torch.bmm(a,memory)
    def forward(self,tokens,memory,mem_mask):
        B,L=tokens.shape; h=memory.new_zeros(1,B,memory.size(-1)); outs=[]
        ctx=memory.masked_fill(mem_mask.unsqueeze(-1),0).sum(1)/(~mem_mask).sum(1,keepdim=True).clamp_min(1)
        for t in range(L):
            z=torch.cat([self.emb(tokens[:,t]),ctx],-1).unsqueeze(1); o,h=self.rnn(z,h)
            ctx=self.attend(o,memory,mem_mask).squeeze(1); outs.append(self.out(torch.cat([o.squeeze(1),ctx],-1)))
        return torch.stack(outs,1)
    def sequence_logprob(self,seqs,memory,mem_mask):
        # seqs: list[list[int]], batch size 1 memory
        vals=[]
        for seq in seqs:
            din=torch.tensor([[BOS]+seq],device=memory.device)
            target=torch.tensor(seq+[EOS],device=memory.device)
            logits=self.forward(din,memory,mem_mask)[0]
            lp=logits.log_softmax(-1).gather(1,target[:,None]).sum()
            vals.append(float(lp))
        return vals

class HybridASR(nn.Module):
    def __init__(self,input_dim,vocab,d=384,layers=4):
        super().__init__(); self.proj=nn.Sequential(nn.LayerNorm(input_dim),nn.Linear(input_dim,d),nn.Dropout(.1))
        self.blocks=nn.ModuleList([ContextBlock(d) for _ in range(layers)])
        self.ctc=nn.Linear(d,vocab); self.decoder=AttentionDecoder(vocab,d)
    def encode(self,x,lengths):
        mask=torch.arange(x.size(1),device=x.device)[None,:]>=lengths[:,None]
        h=self.proj(x)
        for b in self.blocks: h=b(h,mask)
        return h,mask
    def forward(self,x,lengths,dec_in=None):
        h,mask=self.encode(x,lengths); ctc=self.ctc(h)
        aed=None if dec_in is None else self.decoder(dec_in,h,mask)
        return ctc,aed,h,mask


model=HybridASR(INPUT_DIM,len(id_to_char)).to(DEVICE)
model.load_state_dict(ckpt['model'],strict=True)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
del ckpt; gc.collect()
print('20번 Hybrid checkpoint 로딩 완료')

20번 Hybrid checkpoint 로딩 완료


In [5]:
def collapse(path):
    out=[]; prev=None
    for z in path:
        if z!=BLANK and z!=prev: out.append(z)
        prev=z
    return out

def ctc_sequence_logprob(logp,labels):
    # Exact CTC forward score for one collapsed label sequence; logp is [T,V].
    ext=[BLANK]
    for z in labels: ext.extend([int(z),BLANK])
    S=len(ext); neg=logp.new_tensor(-1e30)
    alpha=logp.new_full((S,),-1e30)
    alpha[0]=logp[0,BLANK]
    if S>1: alpha[1]=logp[0,ext[1]]
    for t in range(1,logp.size(0)):
        nxt=logp.new_full((S,),-1e30)
        for s,z in enumerate(ext):
            terms=[alpha[s]]
            if s>0: terms.append(alpha[s-1])
            if s>1 and z!=BLANK and z!=ext[s-2]: terms.append(alpha[s-2])
            nxt[s]=torch.logsumexp(torch.stack(terms),0)+logp[t,z]
        alpha=nxt
    return float(torch.logsumexp(alpha[-2:] if S>1 else alpha[-1:],0))

def ctc_prefix_beam(logp,beam_size=10):
    # logp [T,V], returns (token tuple, acoustic log score)
    beams={(): (0.0,-float('inf'))}
    for t in range(logp.size(0)):
        topv,topi=logp[t].topk(min(logp.size(1),max(beam_size*2,20)))
        nxt={}
        def add(pref,pb,pnb):
            ob,on=nxt.get(pref,(-float('inf'),-float('inf')))
            nxt[pref]=(np.logaddexp(ob,pb),np.logaddexp(on,pnb))
        for pref,(pb,pnb) in beams.items():
            total=np.logaddexp(pb,pnb)
            for val,idx in zip(topv.tolist(),topi.tolist()):
                if idx==BLANK: add(pref,total+val,-float('inf'))
                elif pref and idx==pref[-1]:
                    # 같은 label을 blank 없이 다시 보면 기존 prefix에 머뭅니다.
                    add(pref,-float('inf'),pnb+val)
                    # blank 뒤의 같은 label만 새 반복 문자로 확장됩니다.
                    add(pref+(idx,),-float('inf'),pb+val)
                else: add(pref+(idx,),-float('inf'),total+val)
        beams=dict(sorted(nxt.items(),key=lambda kv:np.logaddexp(*kv[1]),reverse=True)[:beam_size])
    return [(p,float(np.logaddexp(pb,pnb))) for p,(pb,pnb) in sorted(beams.items(),key=lambda kv:np.logaddexp(*kv[1]),reverse=True)[:N_BEST]]

def edit_distance(a,b):
    prev=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        cur=[i]
        for j,y in enumerate(b,1): cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(x!=y)))
        prev=cur
    return prev[-1]

## Hybrid 추론 · 20번 decoding 그대로

원본 CTC prefix beam, exact greedy CTC score 보완, AED 길이 정규화·가중치를 유지합니다. 후보 안의 special token 처리도 원본과 같습니다. 각 샘플의 greedy와 rescored를 동시에 저장하고 중단 시 완료된 샘플부터 재개합니다. shard 전체는 한 번에 하나만 메모리에 올립니다. Reference는 모든 후보가 정해진 후에만 점수 계산에 사용합니다.


In [6]:
ASR_PATH=RUN/'hybrid_transcriptions.csv'
rows=pd.read_csv(ASR_PATH,dtype={'sample_id':str},keep_default_na=False).to_dict('records') if RESUME and ASR_PATH.exists() else []
assert len({r['sample_id'] for r in rows})==len(rows)
done={r['sample_id'] for r in rows}; needed=set(eval_ctc.sample_id)-done
assert done<=set(eval_ctc.sample_id)
canonical={}
for p in FEATURE_DIR.glob('features-*.pt'):
    m=re.fullmatch(r'features-(\d{5})\.pt',p.name)
    if m: canonical[int(m.group(1))]=p
assert set(canonical)==set(range(5)), canonical
meta=eval_ctc.set_index('sample_id')
with torch.inference_mode():
    for shard,path in sorted(canonical.items()):
        if not needed: break
        obj=torch.load(path,map_location='cpu',weights_only=False)
        ids=obj.get('sample_ids',obj.get('ids')); feats=obj.get('features',obj.get('input_features'))
        assert ids is not None and feats is not None and len(ids)==len(feats)
        for sid,feat in tqdm(zip(ids,feats),total=len(ids),desc=f'Hybrid shard {shard}'):
            sid=str(sid)
            if sid not in needed: continue
            x=torch.as_tensor(feat).float()
            assert x.ndim==2 and x.shape[0]>0 and x.shape[1]==INPUT_DIM and torch.isfinite(x).all(), sid
            xl=torch.tensor([len(x)],device=DEVICE); x=x.unsqueeze(0).to(DEVICE)
            with torch.autocast(device_type='cuda',dtype=AMP_DTYPE):
                ctc,_,h,mask=model(x,xl,None)
            lp=ctc[0].float().log_softmax(-1).cpu()
            greedy=collapse(lp.argmax(-1).tolist())
            beams=ctc_prefix_beam(lp,BEAM_SIZE)
            if tuple(greedy) not in {p for p,_ in beams}:
                beams.append((tuple(greedy),ctc_sequence_logprob(lp,greedy)))
            seqs=[list(p) for p,_ in beams]
            with torch.autocast(device_type='cuda',dtype=AMP_DTYPE):
                aed=model.decoder.sequence_logprob(seqs,h,mask)
            scores=[ac/max(1,len(p))+AED_RESCORE_WEIGHT*lm/(max(1,len(p))+1)+LENGTH_BONUS*max(1,len(p))
                    for (p,ac),lm in zip(beams,aed)]
            chosen=seqs[int(np.argmax(scores))]
            rows.append({'sample_id':sid,'greedy_text':decode_text(greedy),
                         'beam1_text':decode_text(seqs[0]),'rescored_text':decode_text(chosen),
                         'nbest_texts':json.dumps([decode_text(z) for z in seqs],ensure_ascii=False)})
            needed.remove(sid)
            atomic_csv(pd.DataFrame(rows),ASR_PATH)
            del x,xl,ctc,h,mask,lp
        del obj,ids,feats; gc.collect()
assert not needed, f'missing features: {sorted(needed)[:10]}'
hybrid=pd.DataFrame(rows)
assert len(hybrid)==300 and hybrid.sample_id.is_unique
inputs=eval_ctc.merge(hybrid,on='sample_id',how='left',sort=False,validate='one_to_one')
assert inputs.sample_id.tolist()==eval_ctc.sample_id.tolist()
atomic_csv(inputs,RUN/'translation_inputs.csv')
del model; gc.collect(); torch.cuda.empty_cache()
print('Hybrid ASR 완료. GPU 메모리 해제 후 LoRA를 로딩합니다.')

Hybrid shard 0:   0%|          | 0/1780 [00:00<?, ?it/s]

Hybrid shard 1:   0%|          | 0/1780 [00:00<?, ?it/s]

Hybrid shard 2:   0%|          | 0/1779 [00:00<?, ?it/s]

Hybrid shard 3:   0%|          | 0/1779 [00:00<?, ?it/s]

Hybrid shard 4:   0%|          | 0/1779 [00:00<?, ?it/s]

Hybrid ASR 완료. GPU 메모리 해제 후 LoRA를 로딩합니다.


In [7]:
asr=[]
for r in inputs.itertuples():
    ref=norm_text(r.transcription_my); denom=max(1,len(ref))
    d={'sample_id':r.sample_id,'source':r.source,'ref_chars':len(ref)}
    for kind,col in [('character_ctc','character_ctc_text'),('greedy','greedy_text'),('beam1','beam1_text'),('rescored','rescored_text')]:
        d[kind+'_edits']=edit_distance(list(ref),list(getattr(r,col)))
        d[kind+'_cer']=d[kind+'_edits']/denom
    d['oracle_nbest_cer']=min(edit_distance(list(ref),list(t))/denom for t in json.loads(r.nbest_texts))
    assert d['oracle_nbest_cer']<=d['greedy_cer']+1e-12
    asr.append(d)
asr=pd.DataFrame(asr); atomic_csv(asr,RUN/'asr_per_sample.csv')
summary=[]
for source,g in [('all',asr)]+list(asr.groupby('source')):
    d={'source':source,'samples':len(g),'oracle_nbest_CER':g.oracle_nbest_cer.mean()}
    for kind in ['character_ctc','greedy','beam1','rescored']:
        d[kind+'_macro_CER']=g[kind+'_cer'].mean()  # 20번과 동일한 sample mean
        d[kind+'_corpus_CER']=g[kind+'_edits'].sum()/max(1,g.ref_chars.sum())
    d['rescoring_gain']=d['greedy_macro_CER']-d['rescored_macro_CER']
    d['ASR_RESCORING_GATE']=bool(d['rescoring_gain']>.003)
    summary.append(d)
asr_metrics=pd.DataFrame(summary); atomic_csv(asr_metrics,RUN/'asr_metrics.csv'); display(asr_metrics)
print('Gate는 20번의 진단 기준입니다. 통과 여부와 관계없이 요청한 번역 비교를 모두 실행합니다.')

,source,samples,oracle_nbest_CER,character_ctc_macro_CER,character_ctc_corpus_CER,greedy_macro_CER,greedy_corpus_CER,beam1_macro_CER,beam1_corpus_CER,rescored_macro_CER,rescored_corpus_CER,rescoring_gain,ASR_RESCORING_GATE
0,all,300,0.117260,0.174476,0.166295,0.135317,0.126699,0.135449,0.125970,0.135744,0.126815,-0.000427,False
1,fleurs,100,0.112258,0.166489,0.165990,0.122204,0.121717,0.121055,0.120769,0.123198,0.122732,-0.000994,False
2,mig-burmese-audio-transcription,100,0.135447,0.207981,0.192373,0.162085,0.153024,0.164183,0.152781,0.161905,0.150352,0.000180,False
3,openslr,100,0.104074,0.148958,0.151909,0.121663,0.121835,0.121109,0.121276,0.122128,0.121695,-0.000465,False


Gate는 20번의 진단 기준입니다. 통과 여부와 관계없이 요청한 번역 비교를 모두 실행합니다.


In [8]:
def standard_prompt(text,lang):
    target='English' if lang=='en' else 'Korean'
    return f'Translate the following Myanmar text into {target}. Preserve its meaning and return only the translation.\n\nMyanmar:\n{text}'
def noisy_prompt(text,lang):
    target='English' if lang=='en' else 'Korean'
    restriction='Use English characters only.' if lang=='en' else 'Do not use Myanmar or Thai script.'
    return (f'Translate the following noisy Myanmar ASR text into {target}.\n'
            'Infer minor recognition errors from context without inventing facts.\n'
            f'Return only one {target} translation. {restriction}\n\nMyanmar:\n{text}')

In [9]:
tokenizer=AutoTokenizer.from_pretrained(str(ADAPTERS['en']),use_fast=True)
tokenizer.padding_side='left'
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
ko_tokenizer=AutoTokenizer.from_pretrained(str(ADAPTERS['ko']),use_fast=True)
assert tokenizer.get_vocab()==ko_tokenizer.get_vocab(), '언어별 tokenizer vocabulary가 다릅니다.'
assert tokenizer.chat_template==ko_tokenizer.chat_template, '언어별 chat template이 다릅니다.'
del ko_tokenizer
base=AutoModelForCausalLM.from_pretrained(LLM_ID,dtype=DTYPE,device_map={'':'cuda:0'},attn_implementation='sdpa')
model=PeftModel.from_pretrained(base,str(ADAPTERS['en']),adapter_name='english',is_trainable=False)
model.load_adapter(str(ADAPTERS['ko']),adapter_name='korean',is_trainable=False)
model.eval()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (english): Dropout(p=0.05, inplace=False)
                  (korean): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (english): Linear(in_features=2560, out_features=16, bias=False)
                  (korean): Linear(in_features=2560, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (english): Linear(in_features=16, out_features=4096, bias=False)
                  (korean): Linear(in_features=16, out_features=4096, bias=False)
    

In [10]:
@torch.inference_mode()
def generate(texts,lang,kind):
    model.set_adapter('english' if lang=='en' else 'korean');fn=standard_prompt if kind=='oracle' else noisy_prompt
    prompts=[tokenizer.apply_chat_template([{'role':'user','content':fn(x,lang)}],tokenize=False,add_generation_prompt=True) for x in texts]
    inp=tokenizer(prompts,return_tensors='pt',padding=True,truncation=True,max_length=768).to(DEVICE)
    out=model.generate(**inp,max_new_tokens=128,do_sample=False,repetition_penalty=1.05,eos_token_id=tokenizer.eos_token_id,pad_token_id=tokenizer.pad_token_id)
    return [x.strip() for x in tokenizer.batch_decode(out[:,inp.input_ids.shape[1]:],skip_special_tokens=True)]

## 18번과 동일한 번역 평가

`oracle`에는 standard prompt, 나머지에는 noisy ASR prompt를 사용합니다. 생성 설정은 `max_new_tokens=128`, greedy generation, `repetition_penalty=1.05`, 입력 최대 768 tokens로 18번과 동일합니다. 매 배치마다 저장하며 같은 실행 hash 안에서 재개합니다. 빈 ASR 입력도 제외하지 않습니다.


In [11]:
CONDITIONS={'oracle':'transcription_my','ctc':'character_ctc_text',
            'hybrid_greedy':'greedy_text','hybrid_beam1':'beam1_text','hybrid_rescored':'rescored_text'}
PRED_PATH=RUN/'translation_predictions.csv'
rows=pd.read_csv(PRED_PATH,dtype={'sample_id':str},keep_default_na=False).to_dict('records') if RESUME and PRED_PATH.exists() else []
keys={(r['sample_id'],r['language'],r['condition']) for r in rows}
assert len(keys)==len(rows)
assert keys<={(sid,lang,kind) for sid in inputs.sample_id for lang in ['en','ko'] for kind in CONDITIONS}
for lang in ['en','ko']:
    for kind,col in CONDITIONS.items():
        todo=inputs[[ (sid,lang,kind) not in keys for sid in inputs.sample_id ]]
        start=0; batch_size=LLM_BATCH_SIZE
        bar=tqdm(total=len(todo),desc=f'{lang} {kind}')
        while start<len(todo):
            batch=todo.iloc[start:start+batch_size]
            try:
                answers=generate(batch[col].tolist(),lang,kind)
            except torch.cuda.OutOfMemoryError:
                gc.collect(); torch.cuda.empty_cache()
                if batch_size==1: raise
                batch_size=max(1,batch_size//2)
                print('OOM: batch size →',batch_size); continue
            assert len(answers)==len(batch)
            for r,p in zip(batch.itertuples(),answers):
                rows.append({'sample_id':r.sample_id,'source':r.source,'language':lang,'condition':kind,
                             'reference':getattr(r,f'translation_{lang}'),'prediction':p})
                keys.add((r.sample_id,lang,kind))
            atomic_csv(pd.DataFrame(rows),PRED_PATH)
            start+=len(batch); bar.update(len(batch))
        bar.close()
result=pd.DataFrame(rows)
assert len(result)==300*2*len(CONDITIONS)
assert result.groupby(['language','condition']).size().eq(300).all()
assert not result.duplicated(['sample_id','language','condition']).any()
# 18번과 동일한 corpus metrics / source 집계
chrf=CHRF(word_order=2); bleu=BLEU(effective_order=True); metrics=[]
for (condition,lang),g in result.groupby(['condition','language']):
    for source,s in [('all',g)]+list(g.groupby('source')):
        metrics.append({'condition':condition,'source':source,'language':lang,'samples':len(s),
            'chrf_pp':chrf.corpus_score(s.prediction.tolist(),[s.reference.tolist()]).score,
            'bleu':bleu.corpus_score(s.prediction.tolist(),[s.reference.tolist()]).score})
metrics=pd.DataFrame(metrics); atomic_csv(metrics,RUN/'translation_metrics.csv'); display(metrics)
(RUN/'metric_signatures.json').write_text(json.dumps({'chrf_pp':str(chrf.get_signature()),'bleu':str(bleu.get_signature())},indent=2))

en oracle:   0%|          | 0/300 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


en ctc:   0%|          | 0/300 [00:00<?, ?it/s]

en hybrid_greedy:   0%|          | 0/300 [00:00<?, ?it/s]

en hybrid_beam1:   0%|          | 0/300 [00:00<?, ?it/s]

...

ko hybrid_greedy:   0%|          | 0/300 [00:00<?, ?it/s]

ko hybrid_beam1:   0%|          | 0/300 [00:00<?, ?it/s]

ko hybrid_rescored:   0%|          | 0/300 [00:00<?, ?it/s]

,condition,source,language,samples,chrf_pp,bleu
0,ctc,all,en,300,28.988688,9.253969
1,ctc,fleurs,en,100,31.330809,9.067992
2,ctc,mig-burmese-audio-transcription,en,100,24.247013,10.999699
3,ctc,openslr,en,100,28.203514,8.013863
4,ctc,all,ko,300,12.083943,3.117568
5,ctc,fleurs,ko,100,12.166590,2.142719
6,ctc,mig-burmese-audio-transcription,ko,100,10.991260,5.754656
7,ctc,openslr,ko,100,12.616528,2.816803
8,hybrid_beam1,all,en,300,32.553582,12.688267
9,hybrid_beam1,fleurs,en,100,36.464899,14.327139


151

In [12]:
# 현재 LoRA를 고정한 front-end 비교: 기존 CTC 대비 Hybrid의 변화
all_metrics=metrics[metrics.source.eq('all')]
comparison=all_metrics.pivot(index='language',columns='condition',values='chrf_pp').reset_index()
comparison['hybrid_gain_vs_character_ctc']=comparison.hybrid_rescored-comparison.ctc
comparison['aed_gain_vs_hybrid_greedy']=comparison.hybrid_rescored-comparison.hybrid_greedy
comparison['gap_to_oracle']=comparison.oracle-comparison.hybrid_rescored
atomic_csv(comparison,RUN/'hybrid_comparison.csv'); display(comparison)
# 18번의 기존 adapter 채택 기준도 동일하게 재현. Hybrid 채택 판정과는 별도입니다.
baseline=pd.DataFrame([{'language':'en','old_ctc':26.609848,'old_oracle':44.980627},
                       {'language':'ko','old_ctc':11.715379,'old_oracle':22.797998}])
now=comparison[['language','ctc','oracle']].rename(columns={'ctc':'new_ctc','oracle':'new_oracle'})
decision=baseline.merge(now,on='language')
decision['ctc_gain']=decision.new_ctc-decision.old_ctc
decision['oracle_change']=decision.new_oracle-decision.old_oracle
decision['pass']=(decision.ctc_gain>=2)&(decision.oracle_change>=-1)
atomic_csv(decision,RUN/'adapter_decision_18_reproduced.csv'); display(decision)
display(result.groupby(['source','language','condition'],group_keys=False).head(2))
print('출력 폴더:',RUN)
print('주요 결과: translation_metrics.csv / hybrid_comparison.csv / asr_metrics.csv')
del model,base; gc.collect(); torch.cuda.empty_cache()

condition,language,ctc,hybrid_beam1,hybrid_greedy,hybrid_rescored,oracle,hybrid_gain_vs_character_ctc,aed_gain_vs_hybrid_greedy,gap_to_oracle
0,en,28.988688,32.553582,32.521782,32.379489,45.323062,3.390801,-0.142293,12.943573
1,ko,12.083943,14.443310,14.403352,14.692800,22.581792,2.608858,0.289448,7.888991


,language,old_ctc,old_oracle,new_ctc,new_oracle,ctc_gain,oracle_change,pass
0,en,26.609848,44.980627,28.988688,45.323062,2.378840,0.342435,True
1,ko,11.715379,22.797998,12.083943,22.581792,0.368564,-0.216206,False


,sample_id,source,language,condition,reference,prediction
0,train_001180,fleurs,en,oracle,Xinhua reported that government investigators ...,State investigators reportedly recovered two B...
1,train_002169,fleurs,en,oracle,A power failure following a routine fire-comma...,A power failure following a routine fire comma...
100,train_006319,mig-burmese-audio-transcription,en,oracle,Calculating that they could make us dance to t...,They calculated that we could secure it by mak...
101,train_006907,mig-burmese-audio-transcription,en,oracle,But in reality...,But in fact...
200,train_005665,openslr,en,oracle,"When we attended university, there was a unive...","When we arrived at university, there was an ar..."
201,train_006056,openslr,en,oracle,I do not want to move to the other side before...,I do not want to go back that way without fini...
300,train_001180,fleurs,en,ctc,Xinhua reported that government investigators ...,The Royal Commission into Abuse of Power revea...
301,train_002169,fleurs,en,ctc,A power failure following a routine fire-comma...,The volcanic activity was triggered by a serie...
400,train_006319,mig-burmese-audio-transcription,en,ctc,Calculating that they could make us dance to t...,They calculated that these were objects of att...
401,train_006907,mig-burmese-audio-transcription,en,ctc,But in reality...,But in fact...


출력 폴더: /content/drive/MyDrive/S2ST/baselines/hybrid_ctc_aed_lora_eval_21/f9f894e6596c1503
주요 결과: translation_metrics.csv / hybrid_comparison.csv / asr_metrics.csv


## 실행 결과 해석

- `ctc`와 `oracle`은 18번의 평가를 현재 저장된 robustness adapter로 재현합니다. 원본 실행의 라이브러리·GPU·adapter 파일 버전이 다르면 수치가 완전히 같지 않을 수 있습니다.
- `hybrid_rescored - ctc`는 같은 LoRA에 연결한 ASR front-end 변경의 번역 효과입니다. `hybrid_rescored - hybrid_greedy`는 AED rescoring 효과입니다.
- `asr_metrics.csv`의 macro CER는 20번의 정의와 같고 corpus CER는 추가 진단입니다. 모델은 20번과 동일하나 padding 영향을 줄이기 위해 ASR을 1개씩 추론하므로 기존 배치 추론과 작은 차이가 날 수 있습니다.
- 이 코드는 기존 저장 모델을 연결하며 새 adapter를 학습하지 않습니다. 한국어 clean adapter를 비교하려면 설정의 `ADAPTERS['ko']`를 `LORA_ROOT/'korean_clean_lora_epoch1'`로 바꾸어 별도 run을 실행하세요. 기본은 요청한 18번과 동일한 robustness adapter입니다.
- checkpoint, cached feature, adapter는 Drive에 이미 있어야 합니다. 이 환경에서는 실제 Drive/GPU 체크포인트를 실행하지 않았으므로 최종 성능은 Colab 실행 후 확인해야 합니다.
